# exp003 — judge Rahul's NullSpaceMap answers against a bf16 unsteered baseline

**What this notebook does.** It does not generate anything. It takes two sets of answers that already exist on Drive
and judges them with the same judge and scorer as exp001 and exp002:

- the **steered** answers from exp_rahul_001 (NullSpaceMap, arm `nullspace_v1`, Qwen3.8-27B in bf16, 98 eval rows);
- the **unsteered** answers that exp002's stage B generated (Qwen3.8-27B in bf16, the 30 rows of the four weak
  honesty categories, with their verdicts already cached).

Both runs used the same rows, chosen by the old row picker (before commit 212e1e4), so the 30 weak-category rows are
shared and can be compared row by row, with the same precision on both sides. That is the like-for-like comparison
exp_rahul_001's write-up was missing: it compared bf16 answers with exp001's 4-bit, pressure-only figures.

**Caveat.** Those 30 rows come from only two source questions per persona category (the old picker's flaw), so this
is a cheap first reading, not a final measurement.

**Runtime: L4.** *Runtime → Run all.* Expected about 20–40 minutes and 1–2 compute units. The other 68 of Rahul's
rows (capability, guard and ceiling-check rows) are judged too and reported per arm, without a paired baseline.

## 1. Settings

In [ ]:
ARM = "nullspace_v1"      # the arm name in Rahul's answers.jsonl
SPLIT = "eval"
STEERED_RUN = "exp_rahul_001_nullspace_map/Qwen3.8-27B_bf16"    # under MyDrive/trust_issues_runs/
UNSTEERED_RUN = "exp002_cast_honesty/Qwen3.8-27B_bf16"
RUN_NAME = "exp003_nullspace_paired_judge/Qwen3.8-27B_bf16"
JUDGE_MODEL = "zai-org/GLM-4-9B-0414"
JUDGE_REVISION = "645b8482494e31b6b752272bf7f7f273ef0f3caf"
JUDGE_BATCH_SIZE = 4      # fits next to the 19 GB judge on an L4's 22 GB; lower to 2 on out-of-memory errors

REPO_URL = "https://github.com/Rupesh-sankhala/Steerability-Trust-issues.git"
REPO_DIR = "/content/Steerability-Trust-issues"

## 2. Check the GPU

In [ ]:
import subprocess

name, memory_mib = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True).stdout.strip().split("\n")[0].split(", ")
memory_gb = int(memory_mib) / 1024
print(f"GPU: {name}, {memory_gb:.0f} GB")
if memory_gb < 20:
    raise SystemExit("The judge needs about 20 GB of GPU memory: choose an L4 (or larger) runtime.")
if memory_gb > 30:
    print("NOTE: this only judges; an L4 is enough and costs fewer units than this GPU.")

## 3. Find the two answer sets on Drive

Rahul's run was on his own Colab, so his `answers.jsonl` must first be copied into
`MyDrive/trust_issues_runs/exp_rahul_001_nullspace_map/Qwen3.8-27B_bf16/` on this account.

In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
RUNS = Path("/content/drive/MyDrive/trust_issues_runs")
STEERED, UNSTEERED, RUN_DIR = RUNS / STEERED_RUN, RUNS / UNSTEERED_RUN, RUNS / RUN_NAME
for needed in (STEERED / "answers.jsonl", UNSTEERED / "answers.jsonl"):
    if not needed.exists():
        raise SystemExit(f"Missing {needed}. Copy it there (see the note above this cell), then Run all again.")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("outputs go to", RUN_DIR)

## 4. Get the team repository (private)

The token is read from Colab's secrets and removed from the clone's settings afterwards, so it is never shown or saved.

In [ ]:
import os
from google.colab import userdata

if not Path(REPO_DIR).exists():
    token = userdata.get("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    result = subprocess.run(["git", "clone", "--depth", "1", authed, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise SystemExit("git clone failed: " + result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 5. Install the scorer's requirements and run its tests

In [ ]:
%pip install -q -r {REPO_DIR}/eval/requirements.txt
!cd {REPO_DIR} && python -m unittest eval/test_scorer.py 2>&1 | tail -3

## 6. Put both arms in one answers file

The scorer compares `--arm` with the `unsteered` arm found in the same file, on the rows both arms answered. Only the
unsteered arm is taken from exp002 (not its CAST arm), and only the unsteered verdicts are carried over, so those 30
answers are not judged again.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

unsteered = [a for a in read_jsonl(UNSTEERED / "answers.jsonl") if a["arm"] == "unsteered"]
steered = [a for a in read_jsonl(STEERED / "answers.jsonl") if a["arm"] == ARM]
if not steered:
    raise SystemExit(f"No answers with arm {ARM!r} in {STEERED / 'answers.jsonl'}: check ARM in section 1.")
shared = {a["row_id"] for a in unsteered} & {a["row_id"] for a in steered}
print(f"{len(unsteered)} unsteered answers, {len(steered)} {ARM} answers, {len(shared)} rows answered by both")
if len(shared) < len({a["row_id"] for a in unsteered}):
    print("WARNING: some unsteered rows have no steered answer; the paired comparison covers only the shared rows.")

combined = RUN_DIR / "combined_answers.jsonl"
combined.write_text("".join(json.dumps(a) + "\n" for a in unsteered + steered))
cache = RUN_DIR / "judge_cache.jsonl"
if not cache.exists() and (UNSTEERED / "judge_cache.jsonl").exists():
    kept = [v for v in read_jsonl(UNSTEERED / "judge_cache.jsonl") if v["key"].startswith("unsteered|")]
    cache.write_text("".join(json.dumps(v) + "\n" for v in kept))
    print(f"{len(kept)} unsteered verdicts carried over from exp002")

## 7. Judge and score

The first call judges every answer not yet in the cache and prints the paired comparison on the shared rows. The
second call reuses those verdicts (no new judging) and prints each arm on all of its rows, which covers Rahul's
capability and guard rows.

In [ ]:
from huggingface_hub import snapshot_download

judge_path = snapshot_download(JUDGE_MODEL, revision=JUDGE_REVISION)
os.environ.update({"JUDGE_BACKEND": "huggingface", "JUDGE_MODEL": judge_path,
                   "JUDGE_BATCH_SIZE": str(JUDGE_BATCH_SIZE)})
print(f"==================== {ARM} vs unsteered (bf16), shared rows")
!cd {REPO_DIR} && python eval/scorer.py --rows data/v7_tidy/{SPLIT}.jsonl --answers {combined} --arm {ARM} \
    --judge --cache {cache} | tee {RUN_DIR}/summary_paired.txt
print(f"\n==================== each arm on all of its rows")
!cd {REPO_DIR} && python eval/scorer.py --rows data/v7_tidy/{SPLIT}.jsonl --answers {combined} \
    --judge --cache {cache} | tee {RUN_DIR}/summary_per_arm.txt

Done. Disconnect the runtime, note the units used, and paste both summaries back. They are also saved on Drive as
`summary_paired.txt` and `summary_per_arm.txt`, next to `judge_cache.jsonl`, which now holds Rahul's verdicts too.